## Data Ingestion for Deep RAG

In this notebook, we'll load extracted data into Qdrant vector database:

- **Markdown**: Page-level chunks with metadata
- **Tables**: Separate documents with context and page numbers
- **Images**: Text descriptions embedded (generated in notebook 06-01b)
- **Hybrid Search**: Dense (semantic) + Sparse (keyword) embeddings

**Prerequisites:**
- Run notebook 06-01 first to extract PDFs
- Run notebook 06-01b to generate image descriptions
- Qdrant server running on localhost:6333
- Google API key set in .env file

**Output:**
- Single Qdrant collection with all content types
- Rich metadata for filtering (company, year, quarter, doc_type, page)
- Deduplication using file hashes

**Make Sure You Have Your QDRANT Vector DB Docker Running**

https://qdrant.tech/

| Point            | **Qdrant** | **Chroma**       | **FAISS** | Weaviate     | Milvus | Pinecone |
| ---------------- | ---------- | ---------------- | --------- | ------------ | ------ | -------- |
| Open Source      | ✅ Yes      | ✅ Yes            | ✅ Yes     | ⚠️ Open-core | ✅ Yes  | ❌ No     |
| DB vs Library    | DB         | DB (dev-focused) | Library   | DB           | DB     | Managed  |
| Hybrid Search    | ✅ Native   | ❌                | ❌         | ✅            | ⚠️     | ✅        |
| Metadata Filter  | ✅ Strong   | ⚠️ Basic         | ❌         | ✅            | ✅      | ✅        |
| Production Ready | ✅ Yes      | ❌ (POC)          | ❌         | ✅            | ✅      | ✅        |
| Local / Offline  | ✅ Yes      | ✅ Yes            | ⚠️        | ⚠️           | ⚠️     | ❌        |


### 0. Qdrant API Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url="https://747a1fab-0ac1-4bff-b62c-b208db10da22.us-west-1-0.aws.cloud.qdrant.io:6333", 
    api_key = os.getenv("QDRANT_API_KEY")
)

print(qdrant_client.get_collections())

C:\Users\choun\AppData\Local\Temp\ipykernel_22616\73740891.py:7: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.1. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(


collections=[CollectionDescription(name='financial_docs')]


In [2]:
qdrant_client = QdrantClient(
    url="http://localhost:6333"
)

print(qdrant_client.get_collections())

collections=[CollectionDescription(name='financial_docs')]


C:\Users\choun\AppData\Local\Temp\ipykernel_22616\3867374210.py:1: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  qdrant_client = QdrantClient(


### 1. Setup and Imports

In [3]:
import hashlib
import time
import uuid
from collections import defaultdict
from pathlib import Path

from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_qdrant import QdrantVectorStore, RetrievalMode, FastEmbedSparse

from langchain_core.documents import Document

### 2. Configuration

In [4]:
# Paths
MARKDOWN_DIR = "data/rag-data/markdown"
TABLES_DIR = "data/rag-data/tables"
IMAGES_DESC_DIR = "data/rag-data/images_desc"

# Qdrant Configuration
COLLECTION_NAME = "financial_docs"
EMBEDDING_MODEL = "models/gemini-embedding-001"

# Ingestion throttling and retry configuration
INGEST_BATCH_SIZE = 8
INGEST_DELAY_SECONDS = 2
MAX_RATE_LIMIT_RETRIES = 5
RETRY_BASE_SECONDS = 60
RETRY_MAX_SECONDS = 300

### 3. Initialize Embeddings and Client

In [5]:
# Embeddings
embeddings = GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

In [6]:
result = embeddings.embed_query('anything')
result


[-0.011239917,
 -0.003467326,
 -0.01105274,
 -0.061294574,
 0.019453146,
 0.0041164085,
 0.027915798,
 0.030927422,
 0.020724343,
 -0.019478088,
 -0.010210239,
 -0.016495619,
 0.006826603,
 0.014704376,
 0.10754771,
 -0.011110156,
 -0.0062106918,
 -0.00084689894,
 -0.014423388,
 -0.012677392,
 0.010252997,
 -0.0056658206,
 0.009224655,
 0.02877456,
 -0.005810222,
 0.005776986,
 0.021359634,
 -0.010408976,
 0.014186839,
 -0.016026922,
 0.018438524,
 0.021181392,
 0.005438517,
 0.0029037222,
 0.017847326,
 0.02250698,
 -0.006939403,
 -0.006773324,
 0.0044366727,
 0.0111732595,
 -0.013067514,
 -0.007899133,
 0.005993952,
 -0.007714994,
 0.008815338,
 0.008485886,
 0.0244984,
 -0.014944867,
 0.0105981305,
 0.024794662,
 -0.016665125,
 -0.016796125,
 -0.019621175,
 -0.15991208,
 -0.022005564,
 0.024612658,
 0.012544701,
 0.00857908,
 0.017164089,
 0.00421856,
 0.0017847299,
 0.002861315,
 -0.009164036,
 -0.020286093,
 -0.012568524,
 0.001471714,
 -0.0022333832,
 0.024866171,
 -0.0022693733,

In [7]:
len(result)

3072

In [8]:
result = sparse_embeddings.embed_query('hi hello')
result

SparseVector(indices=[948991206, 613153351], values=[1.0, 1.0])

In [9]:
result = sparse_embeddings.embed_documents(['hi', 'hello'])
result

[SparseVector(indices=[948991206], values=[1.6877434821696136]),
 SparseVector(indices=[613153351], values=[1.6877434821696136])]

### 4. Create or Recreate Collection

In [ ]:
# # Create vector store at Remote location
'''
vector_store = QdrantVectorStore.from_documents(
      documents=[],
      embedding=embeddings,
      sparse_embedding=sparse_embeddings,
      url="https://747a1fab-0ac1-4bff-b62c-b208db10da22.us-west-1-0.aws.cloud.qdrant.io:6333", 
      api_key = os.getenv("QDRANT_API_KEY"),
      collection_name = COLLECTION_NAME,
      retrieval_mode=RetrievalMode.HYBRID,
      force_recreate=False
 )
 '''

In [10]:
# Create vector store at local computer
vector_store = QdrantVectorStore.from_documents(
    documents=[],
    embedding=embeddings,
    
    sparse_embedding=sparse_embeddings,
    url="http://localhost:6333", 
    collection_name = COLLECTION_NAME,
    retrieval_mode=RetrievalMode.HYBRID,
    force_recreate=False
)

c:\Users\choun\miniconda3\envs\deep-rag\Lib\site-packages\langchain_qdrant\qdrant.py:921: UserWarning: Qdrant client version 1.16.2 is incompatible with server version 1.18.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  client = QdrantClient(**client_options)


In [11]:
vector_store.client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='financial_docs')])

### 5. Helper Functions

In [12]:
def extract_metadata_from_filename(filename: str):
    """
    Extract metadata from filename.
    
    Expected format: CompanyName DocType [Quarter] Year.pdf
    Examples:
        - Amazon 10-Q Q1 2024.pdf
        - Microsoft 10-K 2023.pdf
    """

    filename = filename.replace('.pdf', '').replace('.md', '')
    parts = filename.split()

    return {
        'company_name': parts[0],
        'doc_type': parts[1],
        'fiscal_quarter': parts[2] if len(parts)==4 else None,
        'fiscal_year': parts[-1]
    }

extract_metadata_from_filename('apple 10-k 2023.md')

{'company_name': 'apple',
 'doc_type': '10-k',
 'fiscal_quarter': None,
 'fiscal_year': '2023'}

In [13]:
def compute_file_hash(file_path: Path):

    sha256_hash = hashlib.sha256()

    with open(file_path, 'rb') as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)

    return sha256_hash.hexdigest()


In [24]:
compute_file_hash(Path(r'data\rag-data\markdown\amazon\amazon 10-k 2023.md'))

'fc7817c1b8473b2619bedf24fd8a094d9dbd638ee28546bc9f779937efbfcd1a'

In [25]:
compute_file_hash(Path(r'data\rag-data\markdown\amazon\amazon 10-k 2023 copy.md'))

'fc7817c1b8473b2619bedf24fd8a094d9dbd638ee28546bc9f779937efbfcd1a'

In [30]:
# get the list of ingested file
all_points = vector_store.client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10_000,
    with_payload=True,
    offset=None
)

In [70]:
all_points[1]


'5d635eb5-97bc-422b-ad3f-867f9ecb04ef'

In [ ]:
# if it is first time, you will get None
# all_points[0][0].payload['metadata']['file_hash']
if all_points and len(all_points[0]) > 0:
    file_hash = all_points[0][0].payload.get("metadata", {}).get("file_hash")
else:
    file_hash = None
    print("No point exist yet")
 
print(file_hash)

'455276692b26d8b0d04bcd2eceab403e885b3cc2ec92991b608649bab0956488'

In [14]:
def make_chunk_key(content_type, page):
    page_value = page if page is not None else 'none'
    return f"{content_type}:{page_value}"


def get_ingestion_state():
    ingestion_state = defaultdict(set)
    offset = None

    while True:
        points, offset = vector_store.client.scroll(
                            collection_name=COLLECTION_NAME,
                            limit=10_000,
                            with_payload=True,
                            offset=offset
                        )

        if not points:
            break
        
        for point in points:
            metadata = point.payload.get("metadata", {})
            file_hash = metadata.get("file_hash")
            if file_hash is None:
                continue

            chunk_key = metadata.get("chunk_key") or make_chunk_key(
                metadata.get("content_type", "unknown"),
                metadata.get("page")
            )
            ingestion_state[file_hash].add(chunk_key)

        if offset is None:
            break

    return ingestion_state

In [15]:
ingestion_state = get_ingestion_state()

In [16]:
len(ingestion_state)

45

In [17]:
# extract the page number from the file path
import re

def extract_page_number(file_path: Path):
    pattern = r'page_(\d+)'
    match = re.search(pattern=pattern, string=file_path.stem)
    return int(match.group(1)) if match else None

In [18]:
file_path = Path(r'data\rag-data\images_desc\google\google 10-k 2023\page_28.md')
extract_page_number(file_path)

28

### 6. Ingestion Function

In [19]:
def is_rate_limit_error(error):
    error_message = str(error).upper()
    return "429" in error_message or "RESOURCE_EXHAUSTED" in error_message


def make_point_id(file_hash, chunk_key):
    point_key = f"{COLLECTION_NAME}:{file_hash}:{chunk_key}"
    return str(uuid.uuid5(uuid.NAMESPACE_URL, point_key))


def add_document_batch_with_retry(documents, point_ids):
    for attempt in range(MAX_RATE_LIMIT_RETRIES + 1):
        try:
            vector_store.add_documents(
                documents,
                ids=point_ids,
                batch_size=len(documents)
            )
            return
        except Exception as error:
            if not is_rate_limit_error(error) or attempt == MAX_RATE_LIMIT_RETRIES:
                raise

            wait_seconds = min(
                RETRY_BASE_SECONDS * (2 ** attempt),
                RETRY_MAX_SECONDS
            )
            print(
                f"Gemini rate limit reached. Retrying in {wait_seconds}s "
                f"({attempt + 1}/{MAX_RATE_LIMIT_RETRIES})..."
            )
            time.sleep(wait_seconds)


def ingest_file_in_db(file_path, ingestion_state):
    file_hash = compute_file_hash(file_path)
    path_str = str(file_path)
    if 'markdown' in path_str:
        content_type = 'text'
        doc_name = file_path.name
    elif 'tables' in path_str:
        content_type = 'tables'
        doc_name = file_path.parent.name
    elif 'images_desc' in path_str:
        content_type = 'image'
        doc_name = file_path.parent.name
    else:
        content_type = 'unknown'
        doc_name = file_path.name

    content = file_path.read_text(encoding='utf-8')

    base_metadata = extract_metadata_from_filename(doc_name)

    base_metadata.update({
        'content_type': content_type,
        'file_hash': file_hash,
        'source_file': doc_name
    })

    documents = []
    chunk_keys = []

    if content_type == 'text':
        pages = content.split('<!-- page break -->')
        for idx, page in enumerate(pages, start=1):
            if not page.strip():
                continue

            chunk_key = make_chunk_key(content_type, idx)
            metadata = base_metadata.copy()
            metadata.update({'page': idx, 'chunk_key': chunk_key})
            documents.append(Document(page_content=page, metadata=metadata))
            chunk_keys.append(chunk_key)

    else:
        page_num = extract_page_number(file_path)
        chunk_key = make_chunk_key(content_type, page_num)
        metadata = base_metadata.copy()
        metadata.update({'page': page_num, 'chunk_key': chunk_key})
        documents = [Document(page_content=content, metadata=metadata)]
        chunk_keys = [chunk_key]

    existing_chunks = ingestion_state[file_hash]
    pending_items = [
        (document, chunk_key)
        for document, chunk_key in zip(documents, chunk_keys)
        if chunk_key not in existing_chunks
    ]

    if not pending_items:
        print(f"Already ingested, skipping: {file_path}")
        return "skipped"

    for start in range(0, len(pending_items), INGEST_BATCH_SIZE):
        batch = pending_items[start:start + INGEST_BATCH_SIZE]
        batch_documents = [document for document, _ in batch]
        batch_chunk_keys = [chunk_key for _, chunk_key in batch]
        batch_ids = [
            make_point_id(file_hash, chunk_key)
            for chunk_key in batch_chunk_keys
        ]

        add_document_batch_with_retry(batch_documents, batch_ids)
        existing_chunks.update(batch_chunk_keys)
        time.sleep(INGEST_DELAY_SECONDS)

    print(f"Ingested {len(pending_items)} missing chunk(s): {file_path}")
    return "ingested"


In [20]:
file_path = Path(r'data\rag-data\markdown\amazon\amazon 10-k 2023.md')
ingestion_state = get_ingestion_state()
ingest_file_in_db(file_path, ingestion_state)

Gemini rate limit reached. Retrying in 60s (1/5)...
Gemini rate limit reached. Retrying in 60s (1/5)...
Gemini rate limit reached. Retrying in 60s (1/5)...
Ingested 93 missing chunk(s): data\rag-data\markdown\amazon\amazon 10-k 2023.md


'ingested'

In [26]:
from tqdm import tqdm

base_path = Path('data/rag-data')
all_md_files = list(base_path.rglob("*.md"))

ingestion_state = get_ingestion_state()

for md_file in tqdm(all_md_files):
    ingest_file_in_db(md_file, ingestion_state)

  4%|▍         | 48/1119 [00:00<00:04, 232.35it/s]

Already ingested, skipping: data\rag-data\images_desc\apple\apple 10-k 2023\page_22.md
Already ingested, skipping: data\rag-data\images_desc\apple\apple 10-k 2024\page_23.md
Already ingested, skipping: data\rag-data\images_desc\google\google 10-k 2023\page_28.md
Already ingested, skipping: data\rag-data\images_desc\google\google 10-k 2023\page_29.md
Already ingested, skipping: data\rag-data\images_desc\google\google 10-k 2024\page_29.md
Already ingested, skipping: data\rag-data\images_desc\google\google 10-k 2024\page_30.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-k 2023\page_68.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-k 2023\page_69.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-k 2024\page_64.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-k 2024\page_67.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-k 2024\page_68.md
Already ingested, skipping: data\rag-data\ima

  7%|▋         | 74/1119 [00:00<00:04, 243.65it/s]

Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q2 2025\page_9.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_10.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_11.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_12.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_13.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_2.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_3.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_5.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_7.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_8.md
Already ingested, skipping: data\rag-data\images_desc\meta\meta 10-q q3 2024\page_9.md
Already ingested, skipping: data\rag-da

  8%|▊         | 89/1119 [00:16<05:17,  3.24it/s] 

Ingested 40 missing chunk(s): data\rag-data\markdown\apple\apple 10-k 2024.md
Gemini rate limit reached. Retrying in 60s (1/5)...


... (30 outputs left out)

  9%|▉         | 105/1119 [33:08<18:55:54, 67.21s/it] 

Ingested 18 missing chunk(s): data\rag-data\markdown\meta\meta 10-q q2 2025.md
Gemini rate limit reached. Retrying in 60s (1/5)...
Gemini rate limit reached. Retrying in 120s (2/5)...
Gemini rate limit reached. Retrying in 240s (3/5)...
Gemini rate limit reached. Retrying in 300s (4/5)...
Gemini rate limit reached. Retrying in 300s (5/5)...


  9%|▉         | 105/1119 [50:11<8:04:41, 28.68s/it] 


GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 1000, model: gemini-embedding-1.0\nPlease retry in 49.49965189s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/embed_content_free_tier_requests', 'quotaId': 'EmbedContentRequestsPerDayPerUserPerProjectPerModel-FreeTier', 'qu

### 8. Verify Ingestion

In [108]:
collection_info = vector_store.client.get_collection(COLLECTION_NAME)
collection_info

CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=2867, points_count=2878, segments_count=8, config=CollectionConfig(params=CollectionParams(vectors={'': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, on_disk_payload=True, sparse_vectors={'langchain-sparse': SparseVectorParams(index=None, modifier=None)}), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optim

### 9. Test Search

In [24]:
query = "what is the tesla's revenue"
results = vector_store.similarity_search(query)

In [25]:
results

[Document(metadata={'company_name': 'meta', 'doc_type': '10-q', 'fiscal_quarter': 'q4', 'fiscal_year': '2024', 'content_type': 'image', 'file_hash': '2a23b3453c67ca76f5d4b06dfa4084d3d4d9b2ac403abef2796d63d14592a1d1', 'source_file': 'meta 10-q q4 2024', 'page': 7, 'chunk_key': 'image:7', '_id': '9ce51222-12f1-550b-871b-cd137cac37ff', '_collection_name': 'financial_docs'}, page_content="**Chart Analysis: Net Income (In Millions)**\n\n*   **Metric:** Net Income (in Millions)\n*   **Key Data Points:**\n    *   Q4'22: $4,652\n    *   Q1'23: $5,709\n    *   Q2'23: $7,788\n    *   Q3'23: $11,583\n    *   Q4'23: $14,017\n    *   Q1'24: $12,369\n    *   Q2'24: $13,465\n    *   Q3'24: $15,688\n    *   Q4'24: $20,838\n*   **Significant Trends:** Net Income shows a strong upward trend over the observed period. It more than quadrupled from $4,652 million in Q4'22 to $20,838 million in Q4'24. There was a slight dip in Q1'24 compared to Q4'23, but the overall trajectory is significant growth."),
 Doc